## Ensemble Problema #12 - David Nicolas Torres Marin

Promedio de probabilidades entre dos corridas del mismo workflow (base 7193), cambiando solo la semilla primigenia:

| Semilla | Experimento |
|---|---|
| 119993 | 7301 |
| 299993 | 7302 |

Se promedia prob por numero_de_cliente y se generan cortes de 400 a 1300 (paso 50).

In [ ]:
require("data.table")

EXP_DIR <- "/home/ds/buckets/b1/exp"

experimentos <- c(7301, 7302)


Lectura de las dos corridas

In [ ]:
lista_pred <- list()

for (exp in experimentos) {
  archivo <- file.path(EXP_DIR, paste0("WF", exp), "prediccion.txt")
  if (!file.exists(archivo)) {
    stop(paste("No se encuentra el archivo:", archivo))
  }
  tb <- fread(archivo)
  setnames(tb, "prob", paste0("prob_", exp))
  lista_pred[[as.character(exp)]] <- tb
  cat("Leido", archivo, "-", nrow(tb), "clientes\n")
}


Union y promedio por cliente

In [ ]:
tb_ensemble <- lista_pred[[1]]
for (i in 2:length(lista_pred)) {
  tb_ensemble <- merge(tb_ensemble, lista_pred[[i]], by = "numero_de_cliente")
}

cat("Clientes en el ensemble final:", nrow(tb_ensemble), "\n")

cols_prob <- paste0("prob_", experimentos)
tb_ensemble[, prob := rowMeans(.SD), .SDcols = cols_prob]

summary(tb_ensemble$prob)


Control de integridad

Chequea que no se hayan perdido clientes en el merge y que las probabilidades queden en rango.

In [ ]:
n_esperado <- nrow(lista_pred[[1]])

if (nrow(tb_ensemble) != n_esperado) {
  stop("CONTROL ENSEMBLE: se perdieron clientes en el merge")
}

if (any(is.na(tb_ensemble$prob))) {
  stop("CONTROL ENSEMBLE: hay probabilidades NA luego del promedio")
}

cat("CONTROL ENSEMBLE OK | clientes:", nrow(tb_ensemble),
    "| prob min:", min(tb_ensemble$prob),
    "| prob max:", max(tb_ensemble$prob), "\n")


Grabado del ensemble y generacion de los cortes (400 a 1300)

In [ ]:
carpeta_ensemble <- file.path(EXP_DIR, "WF_ENSEMBLE_7301_7302")
dir.create(carpeta_ensemble, showWarnings = FALSE)
setwd(carpeta_ensemble)

fwrite(tb_ensemble, file = "prediccion_ensemble.txt", sep = "\t")

setorder(tb_ensemble, -prob)

cortes <- seq(400, 1300, by = 50)
dir.create("kaggle", showWarnings = FALSE)

for (envios in cortes) {
  tb_corte <- copy(tb_ensemble[, list(numero_de_cliente)])
  tb_corte[, Predicted := 0L]
  tb_corte[1:envios, Predicted := 1L]

  archivo_salida <- paste0("./kaggle/KA_ENSEMBLE_", envios, ".csv")
  fwrite(tb_corte, file = archivo_salida, sep = ",")
  cat("Generado:", archivo_salida, "\n")
}

cat("Carpeta:", getwd(), "\n")


Envio a Kaggle

Con submit_kaggle en FALSE no se envia nada, solo se generan los CSV.

Para subir: cambiar esa linea a TRUE y correr unicamente esta celda.

In [ ]:
submit_kaggle <- FALSE

competencia <- "utn-2026-virtual-mgr"
semilla_primigenia <- 119993

if (isTRUE(submit_kaggle)) {

  for (envios in cortes) {
    archivo <- paste0("./kaggle/KA_ENSEMBLE_", envios, ".csv")
    mensaje <- paste0("'envios=", envios, "  semilla=", semilla_primigenia, "  ensemble=3'")

    cat("SUBMIT:", archivo, "\n")
    linea <- paste0("kaggle competitions submit -c ", competencia,
                     " -f ", archivo, " -m ", mensaje)
    salida <- system(linea, intern = TRUE)
    cat(salida, sep = "\n")
    cat("\n")
    Sys.sleep(30)
  }

  cat("SUBMITS FINALIZADOS\n")

} else {

  cat("submit_kaggle esta en FALSE. No se envio nada.\n")

}
